# Can we predict whether a warranty claim is filed within a year?
Target: `claim_within_year` (1 = claim filed). Plan: audit the data, explore, build baselines, compare models with cross-validation, evaluate once on a held-out test set, and judge whether the prediction is useful.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
SEED=42
df=pd.read_csv('data.csv'); print(pd.read_csv('columns.csv').to_string(index=False)); df.head()

## 1. Data audit

In [ ]:
print(df.shape); print(df.isna().sum().to_string()); print('duplicate serials:',df.device_serial.duplicated().sum())
print(df.describe().T)
print('daily_hours > 24:',(df.daily_hours>24).sum(),' | == 0:',(df.daily_hours==0).sum())
y=df.claim_within_year; print('base rate:',y.mean().round(4))

Findings: no missing values or duplicates; ~12.6% positives (imbalanced, so accuracy is misleading — we use ROC AUC and PR AUC). `daily_hours` has 23 values above 24, which are impossible for a *daily* figure (self-reported at registration). I keep the rows but clip the feature at 24. `device_serial` is an identifier and is excluded from modelling.

## 2. Exploration: claim rate by feature

In [ ]:
fig,ax=plt.subplots(1,4,figsize=(18,3.5))
for a,col in zip(ax[:3],['device_age_years','price_eur','daily_hours']):
    g=df.groupby(pd.qcut(df[col],8,duplicates='drop'),observed=True).claim_within_year.mean()
    g.plot(marker='o',ax=a); a.set_title(f'claim rate by {col} (octiles)'); a.set_xticklabels([]); a.set_xlabel('')
df.groupby('channel').claim_within_year.mean().plot.bar(ax=ax[3],title='claim rate by channel')
for a in ax: a.axhline(y.mean(),ls='--',c='grey')
plt.tight_layout(); plt.show()
print(df[['device_age_years','price_eur','daily_hours']].corr(method='spearman').round(2))

## 3. Modelling setup
Stratified 80/20 split; the test set is untouched until the final step. Models are compared by 5-fold CV ROC AUC on the training set.

In [ ]:
df['daily_hours']=df.daily_hours.clip(upper=24)
X=df[['channel','device_age_years','price_eur','daily_hours']]
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,stratify=y,random_state=SEED)
num=['device_age_years','price_eur','daily_hours']
def log_price(X): return X.assign(price_eur=np.log1p(X.price_eur))
pre=ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(drop='first'),['channel'])])
models={
 'baseline (prior)':DummyClassifier(strategy='prior'),
 'logistic':Pipeline([('log',FunctionTransformer(log_price)),('pre',pre),('m',LogisticRegression(max_iter=1000))]),
 'grad. boosting':Pipeline([('pre',ColumnTransformer([('c',OneHotEncoder(),['channel'])],remainder='passthrough')),
                           ('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=.05,max_iter=200,l2_regularization=1,random_state=SEED))]),
}
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
for n,m in models.items():
    s=cross_val_score(m,Xtr,ytr,cv=cv,scoring='roc_auc'); print(f'{n:18s} CV AUC {s.mean():.3f} ± {s.std():.3f}')

## 4. Final evaluation on the held-out test set

In [ ]:
best=models['logistic'].fit(Xtr,ytr); p=best.predict_proba(Xte)[:,1]
gb=models['grad. boosting'].fit(Xtr,ytr); pg=gb.predict_proba(Xte)[:,1]
for n,pp in [('logistic',p),('grad. boosting',pg)]:
    print(f'{n:15s} test AUC {roc_auc_score(yte,pp):.3f} | PR-AUC {average_precision_score(yte,pp):.3f} (base rate {yte.mean():.3f}) | Brier {brier_score_loss(yte,pp):.4f} (prior-only {brier_score_loss(yte,np.full(len(yte),ytr.mean())):.4f})')
rng=np.random.default_rng(SEED); yt=yte.values
aucs=[roc_auc_score(yt[i],p[i]) for i in (rng.integers(0,len(yt),len(yt)) for _ in range(500))]
print('logistic test AUC 95% bootstrap CI:',np.percentile(aucs,[2.5,97.5]).round(3))
fig,ax=plt.subplots(1,3,figsize=(16,4))
f,t,_=roc_curve(yte,p); ax[0].plot(f,t); ax[0].plot([0,1],[0,1],'--',c='grey'); ax[0].set_title('ROC (logistic)')
pr,rc,_=precision_recall_curve(yte,p); ax[1].plot(rc,pr); ax[1].axhline(yte.mean(),ls='--',c='grey'); ax[1].set_title('Precision-recall')
fp,mp=calibration_curve(yte,p,n_bins=8,strategy='quantile'); ax[2].plot(mp,fp,'o-'); ax[2].plot([0,.4],[0,.4],'--',c='grey'); ax[2].set_title('Calibration (x=predicted, y=observed)')
plt.tight_layout(); plt.show()

### Lift by risk decile

In [ ]:
d=pd.DataFrame({'p':p,'y':yte.values}); d['decile']=pd.qcut(d.p.rank(method='first'),10,labels=False)+1
t=d.groupby('decile').agg(n=('y','size'),pred=('p','mean'),actual=('y','mean')).round(3); t['lift']=(t.actual/yte.mean()).round(2); print(t)
top=d[d.decile>=9]; print(f'\nTop 20% by predicted risk captures {top.y.sum()/d.y.sum():.0%} of all claims (random targeting: 20%)')

### Drivers

In [ ]:
pi=permutation_importance(best,Xte,yte,scoring='roc_auc',n_repeats=20,random_state=SEED)
print(pd.Series(pi.importances_mean,index=Xte.columns).sort_values(ascending=False).round(4))
print(pd.Series(best.named_steps['m'].coef_[0],index=best.named_steps['pre'].get_feature_names_out()).round(3))

## 5. Conclusion
**Yes, but only moderately.** A simple logistic regression ranks claimants better than chance, but it does not predict individual claims reliably.

- **Ranking skill:** test ROC AUC is about 0.73 (bootstrap 95% CI roughly 0.69–0.77); CV agrees (0.729 ± 0.010). The no-skill baseline is 0.50. PR-AUC is about 0.32 against a 12.6% base rate.
- **Practical value:** the top 20% of devices by predicted risk contain about 43% of all claims (about 2.1x better than random). The top decile claims at about 39%, about 3x the average. The bottom ~40% claim at roughly 3–8%.
- **Individual prediction is weak.** Even the riskiest group is wrong about 60% of the time, so this suits risk scoring and prioritisation (pricing, reserves, monitoring), not a yes/no call on each warranty.
- **Drivers:** older devices at sale (strongest), higher daily use, higher price, and the partner channel (about 14.6% vs about 11.5% for online and store). The features are nearly uncorrelated with each other, and the effects are monotone and roughly additive.
- **Model choice:** gradient boosting did not beat logistic regression (test AUC 0.70 vs 0.73). That suggests there is little non-linear or interaction signal to find with these four features.
- **Calibration:** predicted probabilities track observed rates reasonably, but the test set is small (140 per decile, about 18 positives in the middle ones). Decile-level noise is visible, e.g. decile 9.

**Caveats**
- Only one random split of 7,000 rows, with no temporal split. If the model will score future sales, validate on later data.
- 23 `daily_hours` values above 24 are data errors, so check how they are recorded at the source. I clipped them at 24.
- The "claim within a year" window and the censoring of recent sales are not documented. If recent sales had less than a year to claim, the labels would be biased.
- Gains would most likely come from richer features (product model or category, defect history, region, seller), not from more elaborate models.